# Rotation Visualizer Prototyping

In [63]:
%load_ext autoreload
%autoreload 2

In [103]:
import numpy as np 
from scipy.spatial.transform import Rotation as R
import plotly.graph_objects as go

from src.viz import RotationPlotter, RotationPathPlotter
from src.anim_viz import AnimatedPlotter
from src.kinematics import forward_kinematics
from src.utils import parse_clip

---
## Static Quaternion

In [83]:
theta = np.pi / 2
quat = np.array([1, 1, 1, 1], dtype=float)
quat[:3] = quat[:3] / np.linalg.norm(quat[:3])
quat[:3] = quat[:3] * np.sin(theta / 2)
quat[3:] = quat[3:] * np.cos(theta / 2)
rot_mat = R.from_quat(quat)
rot_mat

Rotation.from_matrix(array([[ 0.33333333, -0.24401694,  0.9106836 ],
                            [ 0.9106836 ,  0.33333333, -0.24401694],
                            [-0.24401694,  0.9106836 ,  0.33333333]]))

In [84]:
axes = np.identity(3)
rotated_axes = rot_mat.apply(axes)

In [85]:
plotter = RotationPlotter(title="Quaternion Rotation Viz Test", up="z", width=800, height=500)
plotter.add_rotation(quat)
plotter.show()

---
## Rotation Path

In [95]:
clip = np.load("data/npz/clip_0.npz", allow_pickle=True)["clip"].item()
clip = parse_clip(clip)
tracks = clip["animationClip"]["tracks"]

In [102]:
plotter = RotationPathPlotter(title="RightHand rotation path", up="y")
plotter.add_rotation_path(tracks["handr.quaternion"]["values"], color_by_time=True)
plotter.show()

---
## Skeleton!

In [119]:
quats = {}
for name, track in tracks.items():
    if track["type"] == "quaternion":
        quats[name.replace(".quaternion", "")] = track["values"]

order, P, bones = forward_kinematics(quats)

ap = AnimatedPlotter(
    title="Test Clip",
    width=800, height=600, 
    margin={"t": 80, "b": 10, "l": 10, "r": 10},
    scene={"aspectmode": "data"},
    scene_camera={"up": {"x": 0, "y": 1, "z": 0}},
    fps=60, stride=4
    )
ap.add_skeleton(P, bones, names=order, joint_size=3, bone_width=4)
ap.show()